## 1. Import Libraries

In this section i import the libraries needed for the project:
- 'pathlib.Path': for file Path handling
- 'pandas': to load and manipulate dataset
- 'scikit-learn': for train/validation/test split
- 'collections.Counter': to check class balance
- 'transformers': to load the pretrained BERT model and tokenizer
- 'transformers.BertTokenizer': to convert text into tokens BERT can process
- 'transformers.BertForSequenceClassification': the pretrained BERT model with a classification head, used for fine-tuning on 3-classes task
- 'torch': for tensor operations and GPU support

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

from transformers import BertTokenizer, BertForSequenceClassification
import torch

In [2]:
BASE_DIR = Path.cwd().parent

## 2.  Import Dataset

I load the CSV file containing the labeled dataset (390 phrases split across the three classes: SOS, MAINTENANCE, SERVICE) and preview the first few rows to confirm it loaded correctly.

In [3]:
df_Path = BASE_DIR / "data"/"raw"/"skyguard_dataset.csv"

df = pd.read_csv(df_Path)

df.head()

,text,label
0,Engineering team confirms the backup generator...,MAINTENANCE
1,"Flight SK315 is experiencing rapid fuel loss, ...",SOS
2,The navigation display is due for scheduled in...,MAINTENANCE
3,Could you check if there's a spare charging ca...,SERVICE
4,Line maintenance has been requested for flight...,MAINTENANCE


## 3. Dataset Quality Check

Before splitting the data, I check:
- the number of samples per class (to confirm balance)
- the presence of duplicate or near-duplicate rows
- any missing/empty values

In [4]:
print("Class distribution:")
print(Counter(df["label"]))

print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate texts only:", df["text"].duplicated().sum())

Class distribution:
Counter({'MAINTENANCE': 130, 'SOS': 130, 'SERVICE': 130})

Missing values:
text     0
label    0
dtype: int64

Exact duplicate rows: 0
Duplicate texts only: 0


## 4. Train/Validation/Test split

I split the dataset into three parts:
- **Train**: used to fine-tune the model
- **Validation**: used to monitor the performance during training
- **Test**: held out completely, used for the final evaluation of the model

The split is stratified by label, so each subset keeps the same class proportions as the original dataset.

In [5]:
train_df, temp_df = train_test_split( 
    df,
    test_size= 0.30,       # First split: Separate 70% of data for training (train_df) and 30% for a temporary set (temp_df)
    stratify= df['label'], # stratify keeps the class proportions identical to the original df
    random_state= 42       # random_state sets a fixed seed to guarantee reproducible splits
       )


val_df, test_df = train_test_split(  
    temp_df,               
    test_size= 0.50,            # Second split: Divide the temporary set equally (50/50) into validation (val_df) and test (test_df) sets      
    stratify= temp_df['label'], # This results in exactly 15% validation and 15% test of the total dataset (df)
    random_state= 42
)

print("Train size: ", len(train_df))
print("Validation size: ", len(val_df))
print("Test size: ", len(test_df))

print('\n')

print("Train class distribution: ", Counter(train_df['label']))
print("Validation class distribution: ", Counter(val_df['label']))
print("Test class distribution: ", Counter(test_df['label']))

Train size:  273
Validation size:  58
Test size:  59


Train class distribution:  Counter({'SOS': 91, 'SERVICE': 91, 'MAINTENANCE': 91})
Validation class distribution:  Counter({'MAINTENANCE': 20, 'SERVICE': 19, 'SOS': 19})
Test class distribution:  Counter({'SOS': 20, 'SERVICE': 20, 'MAINTENANCE': 19})


## 5. Save splits to disk

I save the train, validation, and test sets as separate CSV files in `data/processed/`, so they can be reused without recomputing the split.

In [6]:
train_df.to_csv(BASE_DIR / "data" / "processed" / "train.csv", index= False)
val_df.to_csv(BASE_DIR / "data" / "processed" / "val.csv", index= False)
test_df.to_csv(BASE_DIR / "data" / "processed" / "test.csv", index = False)

print("Files saved successfully in 'data/processed/'")

Files saved successfully in 'data/processed/'


## 6. Load Pretrained Model and Tokenizer

I load 'bert-base-uncased' from Hugging Face, along with its tokenizer.

Since  I have 3 classes (SOS, MAINTENANCE; SERVICE), I configure the model with 'num_label = 3' and a mapping between labels and their numeric IDs.

In [7]:
label2id = {"SOS" : 0, "MAINTENANCE" : 1, "SERVICE" : 2} # maps each class name to a numeric ID
id2label = {0 : "SOS", 1 : "MAINTENANCE", 2 : "SERVICE"} # reverse mapping, numeric ID back to class name

model_name = "bert-base-uncased" # pretrained model to fine-tune

tokenizer = BertTokenizer.from_pretrained(model_name)  # loads the tokenizer matching bert-base-uncased
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 3, # 3 output classes: SOS, MAINTENANCE, SERVICE
    label2id = label2id, # required by Hugging Face to map labels to IDs
    id2label = id2label  # required by Hugging Face to map IDs back to labels
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # use GPU if available, otherwise fall back to CPU

# Explicit equivalent of the inline 'if' statement above for clarity:
# if torch.cuda.is_available():
#     device = torch.device("cuda")
# else:
#     device = torch.device("cpu")

model.to(device) # move model weights to the selected device

print("Model loaded on: ", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on:  cuda
